# 06 · Deployment — Control Strategy S2 (Differential-Drive Arcs)

Per-wheel `set_wheels(left, right)` control with asymmetric speeds: equal
for `straight`, moderate differential for `left`/`right`, aggressive
differential (near-pure rotation) for `hard_left`/`hard_right`. Produced
visibly smoother, more reliable corner completion than S1 (spin-burst) and
was used for the lap-time and per-corner results reported in Sections V-B–V-E.

Requires the ZED SDK (`pyzed`) and a hardware-specific `motors` module — see
the repo README. Not runnable off-robot.

**Note:** this notebook loads `best_model_v3.pth`. The report used **v2**
for all reported deployment results because of its better `right`-class
recall — check which checkpoint you actually want before treating this as
the exact configuration used for the numbers in the report (see README).

In [ ]:
import traitlets
import cv2
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pyzed.sl as sl
import threading
import time
import motors
import ipywidgets.widgets as widgets
from IPython.display import display
from collections import deque
from traitlets.config.configurable import SingletonConfigurable
from torch import nn
from PIL import Image

## Motor diagnostics (run once, before testing the line follower)

Verifies single-motor commands run simultaneously rather than overriding
each other, checks differential drive actually arcs the robot, and measures
the serial control-rate limit of the Yukon per-wheel API (reported as
19 Hz / 52.6 ms per command in Section IV-D).

In [ ]:
robot = motors.MotorsYukon(mecanum=False)

# Test 1: do single-motor commands run simultaneously, or override each other?
print("Test 1: all 4 wheels at 0.1 forward")
print("  expecting: all wheels spin together")
print("  if only back-right spins: commands are overriding")

robot.frontLeft(0.1)
robot.backLeft(0.1)
robot.frontRight(0.1)
robot.backRight(0.1)
time.sleep(2)
robot.stop()
input("  Did all 4 wheels spin together? (press enter to continue) ")

# Test 2: can we do differential drive?
print("\nTest 2: Left wheels stopped, right wheels at 0.15 forward")
print("  robot should rotate left while moving forward (arc left)")

robot.frontLeft(0.0)
robot.backLeft(0.0)
robot.frontRight(0.15)
robot.backRight(0.15)
time.sleep(2)
robot.stop()
input("  Did the robot arc left? (press enter to continue) ")

# Test 3: serial overhead - how fast can we issue 4 motor commands?
print("\nTest 3: timing 50 set_wheels() calls")
start = time.time()
for _ in range(50):
    robot.frontLeft(0.1)
    robot.backLeft(0.1)
    robot.frontRight(0.1)
    robot.backRight(0.1)
elapsed = time.time() - start
robot.stop()
print(f"  50 commands in {elapsed:.2f}s = {elapsed*1000/50:.1f}ms per set_wheels()")
print(f"  Max control rate: {50/elapsed:.1f} Hz")
print(f"  (need at least 5 Hz for usable line following, ideally 15+)")

## Camera

In [ ]:
class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA
        init_params.depth_mode = sl.DEPTH_MODE.ULTRA
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print('Camera open failed:', repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                self.color_value = cv2.cvtColor(self.image.get_data(), cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()


def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])


camera = Camera()

## Load model

In [ ]:
CLASS_NAMES = ['hard_left', 'left', 'straight', 'right', 'hard_right']
MODEL_PATH = 'best_model_v3.pth'  # see note above re: v2 vs v3

device = torch.device('cuda')

model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()
model = model.to(device)

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Model loaded on', device)

## Motor setup — S2: differential-drive arcs

In [ ]:
# arc steering parameters
FORWARD_SPEED = 0.22   # base forward speed
GENTLE_TURN = 0.4      # inside-wheel slowdown for 'left'/'right'
HARD_TURN = 0.95        # inside-wheel slowdown for 'hard_left'/'hard_right'


def set_wheels(left_speed, right_speed):
    robot.frontLeft(left_speed)
    robot.backLeft(left_speed)
    robot.frontRight(right_speed)
    robot.backRight(right_speed)


def arc_left(forward_speed, turn_amount):
    # turn_amount: 0 = straight, 1 = inside wheel stopped
    set_wheels(forward_speed * (1 - turn_amount), forward_speed)


def arc_right(forward_speed, turn_amount):
    set_wheels(forward_speed, forward_speed * (1 - turn_amount))


def execute_command(label):
    if label == 'straight':
        set_wheels(FORWARD_SPEED, FORWARD_SPEED)
    elif label == 'left':
        arc_left(FORWARD_SPEED, GENTLE_TURN)
    elif label == 'hard_left':
        arc_left(FORWARD_SPEED, HARD_TURN)
    elif label == 'right':
        arc_right(FORWARD_SPEED, GENTLE_TURN)
    elif label == 'hard_right':
        arc_right(FORWARD_SPEED, HARD_TURN)

## Callback + live display

Run this cell, then run **START** below.

In [ ]:
straight_idx = CLASS_NAMES.index('straight')
left_idx = CLASS_NAMES.index('left')
right_idx = CLASS_NAMES.index('right')
hl_idx = CLASS_NAMES.index('hard_left')
hr_idx = CLASS_NAMES.index('hard_right')

# smaller buffer, simpler logic - arc steering can react every frame
prediction_buffer = deque(maxlen=3)

display_color = widgets.Image(format='jpeg', width='60%')
display_label = widgets.Label(value='Waiting...')
display(widgets.VBox([display_color, display_label]))

latest_frame = [None]
latest_label_str = ['Waiting...']
display_lock = threading.Lock()


def display_thread_func():
    while True:
        with display_lock:
            frame = latest_frame[0]
            label_str = latest_label_str[0]
        if frame is not None:
            resized = cv2.resize(frame, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
            display_color.value = bgr8_to_jpeg(resized)
            display_label.value = label_str
        time.sleep(0.1)


display_thread = threading.Thread(target=display_thread_func, daemon=True)
display_thread.start()


def on_frame(change):
    frame = change['new']
    if frame is None:
        return

    frame_cropped = frame[200:, :]
    img = Image.fromarray(cv2.cvtColor(frame_cropped, cv2.COLOR_BGR2RGB))
    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(x)
        probs = torch.softmax(output, dim=1)[0]
        pred_id = probs.argmax().item()
        confidence = probs[pred_id].item()

    label = CLASS_NAMES[pred_id]
    if label != 'straight' and confidence < 0.55:
        label = 'straight'
    prediction_buffer.append(label)

    # 3-frame majority vote - kills single-frame noise without much delay
    recent = list(prediction_buffer)
    final = max(set(recent), key=recent.count)

    label_str = (f'HL={probs[hl_idx]:.2f} L={probs[left_idx]:.2f} '
                 f'S={probs[straight_idx]:.2f} R={probs[right_idx]:.2f} '
                 f'HR={probs[hr_idx]:.2f}  |  {final}')

    with display_lock:
        latest_label_str[0] = label_str

    execute_command(final)

## START robot (place on track first)

In [ ]:
prediction_buffer.clear()
camera.observe(on_frame, names=['color_value'])
camera.start()
print('Running')

## STOP robot

In [ ]:
camera.unobserve(on_frame, names=['color_value'])
camera.stop()
robot.stop()
print('Stopped')

## Sanity check (no motors)

In [ ]:
camera.start()
time.sleep(1)
for i in range(10):
    t1 = cv2.getTickCount()
    frame = camera.color_value
    if frame is None:
        print('No frame yet')
        continue
    frame_cropped = frame[200:, :]
    img = Image.fromarray(cv2.cvtColor(frame_cropped, cv2.COLOR_BGR2RGB))
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(x)
        pred_id = output.max(1)[1].item()
    elapsed = (cv2.getTickCount() - t1) / cv2.getTickFrequency()
    print(f'Frame {i+1}: {CLASS_NAMES[pred_id]:12s}  {elapsed*1000:.1f} ms')
camera.stop()
print('Done, no motors moved')